# 第3回B 演習: Forth インタプリタの実装

スライド: [第3回B Forth インタプリタ実装](../03b-forth/)

## 作るもの
スタックベースの言語 **Forth** のインタプリタ。

```
3 4 + .          →  7
: SQUARE DUP * ;
5 SQUARE .       →  25
```

第1回のコマンド表と、第2回のスタック操作・逆ポーランド電卓を土台に、
**字句解析 → ディスパッチループ** という処理系の基本形を組み立てる。

---
## 課題0: 第2回のコードを持ってくる

第2回で書いた**課題4 (スタック操作)** をここに写すこと。
自分で書いたものをそのまま使ってよい。

第2回の**課題5 (ワード表方式の電卓)** も書けているなら、
課題1・課題2 はその続きとして進められる。

In [ ]:
def stack_new():
    return []


def push(s, x):
    s.append(x)


def pop(s):
    if not s:
        raise IndexError('スタックが空です')
    return s.pop()


def peek(s):
    if not s:
        raise IndexError('スタックが空です')
    return s[-1]

---
## 課題1: 算術だけのインタプリタ

空白区切りの文字列を受け取り、スタックを書き換える `forth_eval(stack, text)` を書け。
対応するのは整数と `+ - * /` の4つだけでよい。

- 整数なら push
- 演算子なら2つ pop して計算し、結果を push
- 割り算は整数除算 `//` とする

**ヒント**: `token.lstrip('-').isdigit()` で負数も含めて数値かどうか判定できる。
引き算・割り算は**順序に注意**: 先に pop されるのは右側の値。

In [ ]:
def forth_eval(stack, text):
    for token in text.split():
        # TODO
        pass


s = stack_new()
forth_eval(s, '3 4 + 2 *')
print(s)

In [ ]:
# 確認
def run(text):
    s = stack_new()
    forth_eval(s, text)
    return s

assert run('3 4 +') == [7]
assert run('10 3 -') == [7]        # 順序に注意
assert run('3 4 + 2 *') == [14]
assert run('2 3 4 * +') == [14]
assert run('20 4 /') == [5]
assert run('1 2 3') == [1, 2, 3]
assert run('-5 3 +') == [-2]
print('OK')

---
## 課題2: スタック操作ワード

`forth_eval` に次のワードを追加せよ。**大文字・小文字を区別しない**こと
(`DUP` も `dup` も動く)。

| ワード | 動作 | スタックの変化 |
|---|---|---|
| `.` | 一番上を**出力して**取り除く | (a → ) |
| `DUP` | 一番上を複製 | (a → a a) |
| `DROP` | 一番上を捨てる | (a → ) |
| `SWAP` | 上2つを入れ替え | (a b → b a) |
| `OVER` | 2番目を上にコピー | (a b → a b a) |
| `ROT` | 3番目を一番上へ | (a b c → b c a) |

**ヒント**: 第2回の課題5 と同じ形。`{ワード名: 関数}` の辞書を作り、
`BUILTINS[token](stack)` と呼ぶ。第1回の `@command` によるコマンド表の続きでもある。
この形にしておくと課題3 が楽になる。

In [ ]:
import operator


def binop(f):
    """2引数の関数を「スタックを操作する関数」に変える"""
    def run(stack):
        b = pop(stack)
        a = pop(stack)
        push(stack, f(a, b))
    return run


BUILTINS = {
    '+': binop(operator.add),
    # TODO: - * / と . DUP DROP SWAP OVER ROT を追加
}


def forth_eval(stack, text):
    # TODO: BUILTINS を引いて呼ぶ形に書き換える
    #       知らないワードは ValueError を出すこと
    pass

In [ ]:
# 確認
import io, contextlib

assert run('1 2 SWAP') == [2, 1]
assert run('1 2 swap') == [2, 1]     # 小文字でも動くこと
assert run('5 DUP') == [5, 5]
assert run('1 2 DROP') == [1]
assert run('1 2 OVER') == [1, 2, 1]
assert run('1 2 3 ROT') == [2, 3, 1]
assert run('3 4 + 2 *') == [14]      # 課題1 が壊れていないこと

buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    s = run('42 .')
assert s == [], 'ドットは取り除くこと'
assert '42' in buf.getvalue(), 'ドットは出力すること'

try:
    run('FOO')
except ValueError:
    pass
else:
    raise AssertionError('知らないワードは ValueError を出すこと')
print('OK')

---
## 課題3: ユーザ定義ワード

`: 名前 本体 ;` で新しいワードを定義できるようにせよ。

```
: SQUARE DUP * ;
5 SQUARE .          →  25

: DOUBLE DUP + ;
: QUADRUPLE DOUBLE DOUBLE ;
5 QUADRUPLE .       →  20
```

定義は**辞書に保存**し、その名前が出てきたら本体を評価する。
定義済みのワードを本体から呼べること (`QUADRUPLE` が `DOUBLE` を呼ぶ)。

**ヒント**: `:` に出会ったら、`;` が来るまでのトークンを集めてから辞書に入れる。
そのためにはトークンを**リストにして添字で進める**か、`iter()` にして `next()` で取るとよい。
本体の評価は `forth_eval` を**再帰的に呼ぶ**だけで済む。

In [ ]:
USER_WORDS = {}


def forth_eval(stack, text):
    tokens = text.split()
    i = 0
    while i < len(tokens):
        token = tokens[i]
        # TODO: ':' なら ';' までを USER_WORDS に登録
        #       USER_WORDS にあれば、その本体を評価
        #       それ以外は課題2 までと同じ
        i += 1


USER_WORDS.clear()
s = stack_new()
forth_eval(s, ': SQUARE DUP * ;  5 SQUARE')
print(s)

In [ ]:
# 確認
USER_WORDS.clear()
assert run(': SQUARE DUP * ; 5 SQUARE') == [25]

USER_WORDS.clear()
assert run(': DOUBLE DUP + ; : QUADRUPLE DOUBLE DOUBLE ; 5 QUADRUPLE') == [20]

USER_WORDS.clear()
assert run(': INC 1 + ; 10 INC INC INC') == [13]

# 定義と実行が入り混じっても動くこと
USER_WORDS.clear()
assert run('1 : ADD2 2 + ; ADD2 ADD2') == [5]

# 組み込みが壊れていないこと
USER_WORDS.clear()
assert run('3 4 + 2 *') == [14]
assert run('1 2 3 ROT') == [2, 3, 1]
print('OK')

---
## 課題4: 対話ループ (REPL)

1行入力を受け取って評価し、スタックを表示する対話ループを書け。
`bye` で終了する。

エラーが起きても**ループが落ちない**ようにすること
(第2回の例外処理)。

In [ ]:
def repl():
    stack = stack_new()
    while True:
        line = input('> ')
        # TODO: 'bye' で終了、エラーは表示するだけでループは続ける
        pass


# repl()   ← コメントを外して実行し、いくつか試すこと

### 動かした記録

`repl()` を実行し、**5行以上**入力した結果を残すこと。
ユーザ定義ワードと、わざとエラーを起こす行を必ず含めること。

---
# 発展課題

## 発展1: 比較演算子

`<` `>` `=` を追加せよ。Forth では**真が `-1`、偽が `0`** で表される
(歴史的な理由: すべてのビットが 1)。

In [ ]:
# TODO: BUILTINS に追加


In [ ]:
# 確認
USER_WORDS.clear()
assert run('3 4 <') == [-1]
assert run('4 3 <') == [0]
assert run('3 4 >') == [0]
assert run('4 4 =') == [-1]
assert run('3 4 =') == [0]
print('OK')

## 発展2: 条件分岐

`IF ... ELSE ... THEN` を実装せよ。`IF` の直前にスタックにある値が
0 以外なら `IF` と `ELSE` の間を、0 なら `ELSE` と `THEN` の間を実行する
(`ELSE` は省略できる)。

```
: ABS DUP 0 < IF 0 SWAP - THEN ;
-5 ABS .     →  5
```

**ヒント**: 課題3 と同じく、トークン列の上で対応する `ELSE` / `THEN` を探す。
`IF` が入れ子になりうることに注意 (深さを数える)。

In [ ]:
# TODO


In [ ]:
# 確認
USER_WORDS.clear()
assert run('1 IF 42 THEN') == [42]
assert run('0 IF 42 THEN') == []
assert run('1 IF 10 ELSE 20 THEN') == [10]
assert run('0 IF 10 ELSE 20 THEN') == [20]
assert run(': ABS DUP 0 < IF 0 SWAP - THEN ; -5 ABS') == [5]
assert run(': ABS DUP 0 < IF 0 SWAP - THEN ; 5 ABS') == [5]
# 入れ子
assert run('1 IF 1 IF 7 ELSE 8 THEN ELSE 9 THEN') == [7]
print('OK')

---
## 提出

すべてのセルを実行し、出力を残したまま `.ipynb` をダウンロードして提出。
課題4 の REPL の実行記録も必ず残すこと。